In [ ]:
!pip install pyspark -q
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("S5Clase").getOrCreate()
print(f"✅ Spark {spark.version}")

✅ Spark 4.0.3


In [ ]:
# ============================================================
# CELDA 1: Dataset de pedidos PidelitoApp Lima
# Ejecutar tal cual — no modificar
# ============================================================
import numpy as np
from pyspark.sql import SparkSession

# Crear sesión Spark
spark = SparkSession.builder.appName("S5Clase").getOrCreate()

np.random.seed(42)
N = 800

restaurantes = ["La Mar", "Tanta", "Astrid & Gastón", "El Hornero",
                "Bembos", "KFC Lima", "Pardos Chicken", "Sushi Pop",
                "Cevichería Isolina", "Pizza Hut Miraflores"]
distritos    = ["Miraflores","San Isidro","Barranco","Surco","San Borja",
                "Los Olivos","Ate","SJL","Callao","Villa El Salvador"]
estados      = ["entregado","entregado","entregado","cancelado","en_camino"]

# Convertimos todo a tipos nativos de Python
data = []
for i in range(1, N+1):
    data.append((
        f"PD{i:06d}",                                # str
        str(np.random.choice(restaurantes)),         # str
        str(np.random.choice(distritos)),            # str
        int(np.random.randint(7, 23)),               # int
        float(round(np.random.uniform(18, 120), 2)), # float
        float(round(np.random.uniform(3, 12), 2)),   # float
        int(np.random.randint(15, 65)),              # int
        float(round(np.random.normal(4.2, 0.5), 1)), # float
        f"REP{np.random.randint(1,40):03d}",         # str
        str(np.random.choice(estados, p=[.75,.08,.05,.10,.02])) # str
    ))

# Crear DataFrame con esquema explícito
from pyspark.sql.types import *

schema = StructType([
    StructField("id_pedido", StringType(), True),
    StructField("restaurante", StringType(), True),
    StructField("distrito", StringType(), True),
    StructField("hora", IntegerType(), True),
    StructField("monto_soles", FloatType(), True),
    StructField("costo_delivery", FloatType(), True),
    StructField("tiempo_entrega_min", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("id_repartidor", StringType(), True),
    StructField("estado", StringType(), True)
])

pedidos = spark.createDataFrame(data, schema)

pedidos.createOrReplaceTempView("pedidos")
print(f"✅ {pedidos.count()} pedidos cargados")
pedidos.show(3)


✅ 800 pedidos cargados
+---------+--------------+----------+----+-----------+--------------+------------------+------+-------------+---------+
|id_pedido|   restaurante|  distrito|hora|monto_soles|costo_delivery|tiempo_entrega_min|rating|id_repartidor|   estado|
+---------+--------------+----------+----+-----------+--------------+------------------+------+-------------+---------+
| PD000001|Pardos Chicken|     Surco|  19|      36.71|         10.02|                35|   4.1|       REP011|entregado|
| PD000002|        Bembos|     Surco|  14|      90.22|          3.19|                16|   4.1|       REP024|entregado|
| PD000003|      KFC Lima|San Isidro|  22|     119.21|          8.56|                36|   3.0|       REP028|cancelado|
+---------+--------------+----------+----+-----------+--------------+------------------+------+-------------+---------+
only showing top 3 rows


In [ ]:
print("Columnas:", pedidos.columns)
print("Total registros:", pedidos.count())
spark.catalog.listTables()


Columnas: ['id_pedido', 'restaurante', 'distrito', 'hora', 'monto_soles', 'costo_delivery', 'tiempo_entrega_min', 'rating', 'id_repartidor', 'estado']
Total registros: 800


[Table(name='pedidos', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [ ]:
spark.catalog.listTables()


[Table(name='pedidos', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [ ]:
top_restaurantes = spark.sql("""
    SELECT
        restaurante,
        COUNT(*)                        AS total_pedidos,
        ROUND(SUM(monto_soles), 2)      AS ingresos_totales,
        ROUND(AVG(rating), 2)           AS rating_prom,
        ROUND(AVG(tiempo_entrega_min),1)AS tiempo_entrega_prom
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY restaurante
    ORDER BY ingresos_totales DESC
    LIMIT 5
""")

print("🍽️ TOP 5 RESTAURANTES — INGRESOS DEL MES:")
top_restaurantes.show(truncate=False)


🍽️ TOP 5 RESTAURANTES — INGRESOS DEL MES:
+--------------------+-------------+----------------+-----------+-------------------+
|restaurante         |total_pedidos|ingresos_totales|rating_prom|tiempo_entrega_prom|
+--------------------+-------------+----------------+-----------+-------------------+
|Tanta               |86           |6280.6          |4.28       |37.4               |
|Pardos Chicken      |85           |5702.74         |4.28       |41.6               |
|La Mar              |83           |5643.05         |4.17       |39.1               |
|Pizza Hut Miraflores|81           |5475.86         |4.24       |40.9               |
|KFC Lima            |71           |4793.57         |4.13       |38.1               |
+--------------------+-------------+----------------+-----------+-------------------+



In [ ]:
repartidores_eficientes = spark.sql("""
    SELECT
        id_repartidor,
        COUNT(*)                                AS pedidos_entregados,
        ROUND(AVG(tiempo_entrega_min), 1)       AS tiempo_prom,
        ROUND(AVG(rating), 2)                   AS rating_prom,
        ROUND(SUM(costo_delivery), 2)           AS ingresos_delivery
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY id_repartidor
    HAVING COUNT(*) > 10
       AND AVG(tiempo_entrega_min) < 40
       AND AVG(rating) > 4.0
    ORDER BY rating_prom DESC
""")

print("⚡ REPARTIDORES EFICIENTES (>10 pedidos, tiempo<40min, rating>4.0):")
repartidores_eficientes.show(10)
print(f"Total repartidores que cumplen los criterios: {repartidores_eficientes.count()}")


⚡ REPARTIDORES EFICIENTES (>10 pedidos, tiempo<40min, rating>4.0):
+-------------+------------------+-----------+-----------+-----------------+
|id_repartidor|pedidos_entregados|tiempo_prom|rating_prom|ingresos_delivery|
+-------------+------------------+-----------+-----------+-----------------+
|       REP037|                17|       39.6|       4.43|           123.73|
|       REP031|                14|       38.1|       4.39|            96.85|
|       REP022|                12|       39.8|       4.38|           107.65|
|       REP016|                14|       39.5|       4.36|            95.27|
|       REP004|                22|       38.4|        4.3|           162.79|
|       REP029|                23|       34.0|       4.29|           163.63|
|       REP014|                16|       35.3|       4.29|           120.01|
|       REP034|                17|       39.5|       4.27|            149.4|
|       REP015|                14|       37.8|       4.27|            86.35|
|       R

In [ ]:
pedidos.createOrReplaceTempView("pedidos")
print(f"✅ {pedidos.count()} pedidos cargados")
pedidos.show(3)
demanda_hora = spark.sql("""
    SELECT
        hora,
        COUNT(*)                    AS total_pedidos,
        ROUND(SUM(monto_soles), 2)  AS ingresos_hora,
        ROUND(AVG(tiempo_entrega_min), 1) AS tiempo_prom
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY hora
    ORDER BY total_pedidos DESC
""")

print("🕐 DEMANDA POR HORA DEL DÍA:")
demanda_hora.show(8)
demanda_hora = spark.sql("""
    SELECT
        hora,
        COUNT(*) AS total_pedidos,
        ROUND(SUM(monto_soles), 2) AS ingresos_hora,
        ROUND(AVG(tiempo_entrega_min), 1) AS tiempo_prom
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY hora
    ORDER BY total_pedidos DESC
""")
demanda_hora.show(8)


✅ 800 pedidos cargados
+---------+--------------+----------+----+-----------+--------------+------------------+------+-------------+---------+
|id_pedido|   restaurante|  distrito|hora|monto_soles|costo_delivery|tiempo_entrega_min|rating|id_repartidor|   estado|
+---------+--------------+----------+----+-----------+--------------+------------------+------+-------------+---------+
| PD000001|Pardos Chicken|     Surco|  19|      36.71|         10.02|                35|   4.1|       REP011|entregado|
| PD000002|        Bembos|     Surco|  14|      90.22|          3.19|                16|   4.1|       REP024|entregado|
| PD000003|      KFC Lima|San Isidro|  22|     119.21|          8.56|                36|   3.0|       REP028|cancelado|
+---------+--------------+----------+----+-----------+--------------+------------------+------+-------------+---------+
only showing top 3 rows
🕐 DEMANDA POR HORA DEL DÍA:
+----+-------------+-------------+-----------+
|hora|total_pedidos|ingresos_hora|tiem

# Pregunta 1:¿Cuándo hay más pedidos según tu consulta (hora del día)?¿Coincide con lo que esperabas para Lima? ¿Por qué sí o no?
Se observa que La mayor cantidad de pedidos ocurre en las horas de almuerzo que  es en el periodo de 12 a 13 Horas y la cena de 19 a 21.
Motivo por lo que coincide con lo esperado en Lima porque son los horarios habituales en que las personas piden delivery.
Refleja patrones culturales de consumo: almuerzo con mayor auje al mediodía y cena ligera en la noche.

# Pregunta 2:
#Si la empresa quiere bonificar a los mejores repartidores,
#¿cuál de los 3 criterios de la Consulta 3 es el más importante?Justifica con un argumento de negocio.

Se puede observar que el criterio más importante es el rating > 4.0.
Desde el punto de vista de negocio, un excelente  servicio al cliente asegura una buena satisfacción y se obtine clientes fieles, y asi se multiplica experiencias y recomendaciones, lo que genera más pedidos futuros y reputación positiva para la empresa.
El tiempo de entrega y la cantidad de pedidos son relevantes, pero sin calidad percibida (rating),el cliente no vuelve a pedir.


# Pregunta 3:
#¿Por qué usamos Spark SQL y no pandas para este análisis? Da UN argumento técnico específico.

En esta oprtunidad hemos utilizado Spark SQL en lugar de pandas porque Spark distribuye el procesamiento en múltiples nodos,permitiendo manejar datasets grandes sin problemas de memoria.
Pandas trabaja en memoria local y se limita al tamaño de la RAM,
mientras que Spark escala horizontalmente en clusters.

In [ ]:
# Consulta correcta en Colab usando Spark SQL
consulta = spark.sql("""
    SELECT id_repartidor, COUNT(*) AS pedidos
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY id_repartidor
    HAVING COUNT(*) > 10
""")

consulta.show(5)


+-------------+-------+
|id_repartidor|pedidos|
+-------------+-------+
|       REP019|     22|
|       REP037|     17|
|       REP035|     21|
|       REP025|     24|
|       REP034|     17|
+-------------+-------+
only showing top 5 rows


In [ ]:
consulta_having = spark.sql("""
    SELECT id_repartidor, COUNT(*) AS pedidos
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY id_repartidor
    HAVING COUNT(*) > 10
""")
consulta_having.show(5)


+-------------+-------+
|id_repartidor|pedidos|
+-------------+-------+
|       REP019|     22|
|       REP037|     17|
|       REP035|     21|
|       REP025|     24|
|       REP034|     17|
+-------------+-------+
only showing top 5 rows


In [ ]:
consulta_cancelados = spark.sql("""
    SELECT id_pedido, estado, restaurante
    FROM pedidos
    WHERE estado = 'entregado' OR estado = 'cancelado'
    LIMIT 10
""")
consulta_cancelados.show()


+---------+---------+------------------+
|id_pedido|   estado|       restaurante|
+---------+---------+------------------+
| PD000001|entregado|    Pardos Chicken|
| PD000002|entregado|            Bembos|
| PD000003|cancelado|          KFC Lima|
| PD000004|entregado|   Astrid & Gastón|
| PD000005|entregado|             Tanta|
| PD000006|entregado|            La Mar|
| PD000007|entregado|             Tanta|
| PD000008|cancelado|Cevichería Isolina|
| PD000009|entregado|Cevichería Isolina|
| PD000010|entregado|            Bembos|
+---------+---------+------------------+



In [ ]:
consulta_avg = spark.sql("""
    SELECT id_repartidor, AVG(rating) AS r
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY id_repartidor
    HAVING r > 4.0
""")
consulta_avg.show(5)


+-------------+------------------+
|id_repartidor|                 r|
+-------------+------------------+
|       REP019| 4.340909069234675|
|       REP037| 4.429411789950202|
|       REP035| 4.004761877514067|
|       REP025|   4.1791666050752|
|       REP034|4.2705882857827575|
+-------------+------------------+
only showing top 5 rows
